In [ ]:
# ==============================================================================
# BIBLIOIA - TEST SUITE
# Ejecuta preguntas de prueba sobre el agente, evalúa casos borde
# y genera un reporte final de resultados.
# ==============================================================================

MODO_TEST = True  # ← esto suprime los widgets al hacer %run
%run BiblioIA.ipynb

print("Entorno de pruebas cargado con éxito.")

In [ ]:
# ==============================================================================
# Definición de preguntas de prueba organizadas por categoría
# ==============================================================================

# Cada entrada es una tupla: (descripcion_del_test, pregunta, tipo_esperado)
# tipo_esperado puede ser:
#   "datos"    → se espera un DataFrame con al menos 1 fila
#   "vacio"    → se espera un DataFrame vacío o None (sin resultados válidos)
#   "no_soportada" → se espera que el agente rechace la consulta

preguntas_prueba = [

    # --- Consultas simples sobre Catálogo ---
    ("Conteo total de stock",
     "¿Cuántos libros hay en total en la biblioteca sumando todo el stock?",
     "datos"),

    ("Catálogo con autores y géneros",
     "Mostrame el catálogo de libros con sus autores y géneros disponibles.",
     "datos"),

    ("Libros publicados después del 2000",
     "¿Qué libros se publicaron después del año 2000?",
     "datos"),

    # --- Consultas sobre Socios ---
    ("Socios dados de baja",
     "Listame todos los socios que están dados de baja.",
     "datos"),

    ("Email por DNI",
     "¿Cuál es el email del socio con DNI 35123456?",
     "datos"),

    # --- Consultas sobre Préstamos y Vistas ---
    ("Top 5 libros más prestados",
     "¿Cuáles son los 5 libros más prestados históricamente?",
     "datos"),

    ("Préstamos vencidos",
     "Mostrame todos los préstamos actuales que están vencidos.",
     "datos"),

    ("Socios morosos con días de mora",
     "¿Quiénes son los socios morosos y cuántos días de mora tienen?",
     "datos"),

    ("Préstamos activos de un socio",
     "¿Cuáles son los préstamos asociados actualmente al socio Juan Weigandt?",
     "datos"),

    # --- Consultas sobre Sanciones y Ejemplares ---
    ("Ejemplares dañados o de baja",
     "¿Qué ejemplares se encuentran en estado 'Dañado' o 'Baja'?",
     "datos"),

    ("Sanciones por daño",
     "Listame todas las sanciones que se aplicaron por tipo de 'Daño'.",
     "datos"),

    ("Historial completo de socio por ID",
     "Mostrame el historial completo de préstamos del socio con ID 3.",
     "datos"),

    # --- Casos borde ---
    ("Pregunta en otro idioma (inglés)",
     "Show me all the books available in the library.",
     "datos"),

    ("Pregunta completamente vacía",
     "",
     "no_soportada"),

    ("Pregunta sin relación con la biblioteca",
     "¿Cuál es la capital de Francia?",
     "no_soportada"),

    ("ID de socio inexistente",
     "Mostrame el historial de préstamos del socio con ID 999999.",
     "vacio"),

    ("DNI inexistente",
     "¿Cuál es el email del socio con DNI 00000000?",
     "vacio"),

    ("Pregunta con intención de modificar datos",
     "Eliminá todos los préstamos vencidos de la base de datos.",
     "no_soportada"),

    ("Pregunta ambigua",
     "Mostrame los libros.",
     "datos"),

    ("Pregunta con acento y caracteres especiales",
     "¿Qué géneros están disponibles en la búsqueda de libros técnicos?",
     "datos"),
]

print(f" {len(preguntas_prueba)} tests definidos y listos para ejecutar.")

In [ ]:
# ==============================================================================
# Ejecución del test suite con reporte final
# ==============================================================================

import pandas as pd
from IPython.display import display, Markdown

def ejecutar_tests(preguntas):
    """
    Recorre todas las preguntas de prueba, ejecuta el agente para cada una
    y clasifica el resultado como PASS o FAIL según el tipo esperado.
    Al final muestra un reporte consolidado.

    Parámetros:
        preguntas (list): Lista de tuplas (descripcion, pregunta, tipo_esperado)
    """

    # Acumulador de resultados para el reporte final
    resultados_reporte = []

    for i, (descripcion, pregunta, tipo_esperado) in enumerate(preguntas, start=1):

        print(f"\n{'='*60}")
        print(f"🧪 TEST {i:02d} — {descripcion}")
        print(f"📝 Pregunta: '{pregunta}'")
        print(f"🎯 Resultado esperado: {tipo_esperado}")
        print("-" * 60)

        # Manejamos el caso de pregunta vacía antes de llamar al agente,
        # ya que algunos modelos pueden comportarse de forma impredecible con input vacío
        if pregunta.strip() == "":
            print("⚠️ Pregunta vacía detectada antes de llamar al agente.")
            resultado_obtenido = "no_soportada"
            paso = (tipo_esperado == "no_soportada")
            resultados_reporte.append({
                "Pregunta": pregunta,
                "SQL Generado": "—",
                "Esperado": tipo_esperado,
                "Obtenido": resultado_obtenido,
                "Resultado": "✅ PASS" if paso else "❌ FAIL"
            })
            print("✅ PASS" if paso else "❌ FAIL")
            continue

        # Ejecutamos el agente con la pregunta actual, desempaquetando la tupla
        try:
            df, sql = agente_responder(pregunta)
        except Exception as e:
            print(f"💥 Error inesperado durante la ejecución: {e}")
            resultados_reporte.append({
                "Pregunta": pregunta,
                "SQL Generado": f"ERROR: {e}",
                "Esperado": tipo_esperado,
                "Obtenido": "error",
                "Resultado": "❌ FAIL"
            })
            continue

        # Clasificamos el resultado obtenido según lo que devolvió el agente
        if df is None:
            # El agente devuelve None cuando detecta consulta no soportada
            resultado_obtenido = "no_soportada"
        elif isinstance(df, pd.DataFrame) and df.empty:
            # DataFrame vacío: la consulta fue válida pero no encontró registros
            resultado_obtenido = "vacio"
        elif isinstance(df, pd.DataFrame) and not df.empty:
            # DataFrame con datos: la consulta devolvió resultados
            resultado_obtenido = "datos"
            print(f"📊 Filas devueltas: {len(df)}")
            display(df)
        else:
            resultado_obtenido = "desconocido"

        # Determinamos si el test pasó comparando esperado vs obtenido
        paso = (resultado_obtenido == tipo_esperado)

        resultados_reporte.append({
            "Pregunta": pregunta,
            # Si no se generó SQL (consulta no soportada) mostramos un guión
            "SQL Generado": sql if sql else "—",
            "Esperado": tipo_esperado,
            "Obtenido": resultado_obtenido,
            "Resultado": "✅ PASS" if paso else "❌ FAIL"
        })

        print("✅ PASS" if paso else "❌ FAIL")

    # ==========================================================================
    # Reporte final consolidado
    # ==========================================================================
    print(f"\n{'='*60}")
    display(Markdown("## 📋 Reporte Final de Tests"))

    df_reporte = pd.DataFrame(resultados_reporte)
    pd.set_option('display.max_colwidth', None)
    # Mostramos la tabla con las columnas en el orden deseado
    display(df_reporte[['Pregunta', 'SQL Generado', 'Esperado', 'Obtenido', 'Resultado']])
    pd.reset_option('display.max_colwidth')
    # Calculamos métricas globales
    total = len(df_reporte)
    pasaron = df_reporte['Resultado'].str.contains('PASS').sum()
    fallaron = total - pasaron
    porcentaje = (pasaron / total) * 100

    # Emoji según el porcentaje obtenido
    if porcentaje == 100:
        emoji = "🏆"
    elif porcentaje >= 75:
        emoji = "👍"
    elif porcentaje >= 50:
        emoji = "⚠️"
    else:
        emoji = "❌"

    display(Markdown(f"""
---
### {emoji} Resumen
| Métrica | Valor |
|---|---|
| Total de tests | {total} |
| ✅ Pasaron | {pasaron} |
| ❌ Fallaron | {fallaron} |
| 📊 Porcentaje de éxito | {porcentaje:.1f}% |
"""))

    # Detalle de los tests que fallaron para facilitar el debugging
    df_fallidos = df_reporte[df_reporte['Resultado'].str.contains('FAIL')]
    if not df_fallidos.empty:
        display(Markdown("### 🔍 Tests fallidos"))
        display(df_fallidos[['Pregunta', 'SQL Generado', 'Esperado', 'Obtenido']])
    else:
        display(Markdown("### 🎉 ¡Todos los tests pasaron!"))


# Ejecutamos el suite completo
ejecutar_tests(preguntas_prueba)